<a href="https://colab.research.google.com/github/akul-bharadwaj/various-agents/blob/main/Case_Study_1_Basic_Agentic_System_Fintech_Loan_Advisor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Case Study 1: Basic Agentic System — Fintech Loan Advisor

## Objective

Build a basic agentic system for a fintech loan-advisory use case.

The agent must:

- take a loan application;
- identify missing information;
- use the **OpenAI API** to reason about the next action;
- call a mock credit-score tool;
- calculate risk through a deterministic Python tool;
- approve or reject the application;
- update its internal state; and
- continue iterating until a final decision is produced.

The implementation uses the **OpenAI Python SDK and Responses API**, but no agent framework such as LangChain, CrewAI, or the OpenAI Agents SDK.

> **Educational disclaimer:** The rules and thresholds used here are simplified examples. They must not be used for real lending, credit, or eligibility decisions.

## Agent architecture

```text
                    ┌─────────────────────────┐
                    │    Loan application     │
                    └────────────┬────────────┘
                                 │
                                 ▼
                    ┌─────────────────────────┐
                    │       observe()         │
                    │ Inspect data and state  │
                    └────────────┬────────────┘
                                 │
                                 ▼
                    ┌─────────────────────────┐
                    │        reason()         │
                    │ OpenAI selects one tool │
                    └────────────┬────────────┘
                                 │
                                 ▼
                    ┌─────────────────────────┐
                    │          act()          │
                    │ Execute Python function │
                    └────────────┬────────────┘
                                 │
                                 ▼
                    ┌─────────────────────────┐
                    │     update_state()      │
                    │ Store tool result/data  │
                    └────────────┬────────────┘
                                 │
                   Decision? ────┴──── No ────┐
                       │                      │
                      Yes                     └── Repeat
                       │
                       ▼
                    Approve / Reject
```

### Separation of responsibilities

- **OpenAI model:** decides which action should occur next.
- **Python tools:** retrieve the mock score, calculate risk, and apply the approval rules.
- **Agent state:** stores the application, intermediate outputs, history, and final decision.
- **Agent loop:** repeatedly calls `observe() → reason() → act() → update_state()`.

## 1. Install the OpenAI Python package

Run this cell once in Google Colab.

In [1]:
!pip -q install --upgrade openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 53.6 MB/s eta 0:00:00


## 2. Configure the API key

The key is entered securely with `getpass`, so it is not printed in the notebook output.

Do not hard-code an API key in a notebook that will be shared or uploaded to GitHub.

In [2]:
import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("API key configured.")

Enter your OpenAI API key: ··········
API key configured.


## 3. Imports and OpenAI client

`gpt-4o-mini` is used as a cost-conscious default. It can be replaced with another tool-calling model available to your API project.

In [3]:
import json
from pprint import pprint
from typing import Any, Callable, Dict, List, Optional

from openai import OpenAI

MODEL = "gpt-4o-mini"
client = OpenAI()

print("Client created.")
print("Selected model:", MODEL)

Client created.
Selected model: gpt-4o-mini


## 4. Create the mock tools

The model does **not** calculate the credit score or lending risk itself. It selects an action, and `act()` executes one of these controlled Python functions.

### Risk-scoring rules

A larger score means greater risk.

| Factor | Condition | Points |
|---|---:|---:|
| Credit score | Below 580 | +55 |
|  | 580–669 | +35 |
|  | 670–739 | +18 |
|  | 740 or above | +5 |
| Loan-to-income ratio | Above 0.80 | +35 |
|  | Above 0.50 | +22 |
|  | Above 0.30 | +12 |
|  | 0.30 or below | +5 |
| Employment history | Below 1 year | +20 |
|  | Below 3 years | +10 |
|  | 3 years or above | +3 |

Decision rules:

- age below 21 → reject;
- risk score of 45 or below → approve;
- risk score above 45 → reject.

In [4]:
MOCK_CREDIT_DATABASE = {
    "USER1001": 780,
    "USER1002": 690,
    "USER1003": 540,
    "USER1004": 720,
}


def get_credit_score(user_id: str) -> Dict[str, Any]:
    """Return a mock credit score for a supplied user ID."""
    if not isinstance(user_id, str) or not user_id.strip():
        raise ValueError("A valid user_id is required.")

    normalized_id = user_id.strip().upper()

    if normalized_id in MOCK_CREDIT_DATABASE:
        score = MOCK_CREDIT_DATABASE[normalized_id]
        source = "mock_database"
    else:
        # Produces a repeatable score for any unknown ID.
        deterministic_value = sum(ord(character) for character in normalized_id)
        score = 550 + (deterministic_value % 250)
        source = "deterministic_mock_generator"

    return {
        "user_id": normalized_id,
        "credit_score": score,
        "source": source,
    }


def calculate_risk(
    application: Dict[str, Any],
    credit_score: int,
) -> Dict[str, Any]:
    """Calculate a deterministic risk score from the application."""
    age = int(application["age"])
    annual_income = float(application["annual_income"])
    loan_amount = float(application["loan_amount"])
    employment_years = float(application["employment_years"])

    if age <= 0:
        raise ValueError("age must be greater than 0.")
    if annual_income <= 0:
        raise ValueError("annual_income must be greater than 0.")
    if loan_amount <= 0:
        raise ValueError("loan_amount must be greater than 0.")
    if employment_years < 0:
        raise ValueError("employment_years cannot be negative.")
    if not 300 <= int(credit_score) <= 850:
        raise ValueError("credit_score must be between 300 and 850.")

    risk_score = 0
    risk_factors: List[str] = []

    # Credit-score component
    if credit_score < 580:
        risk_score += 55
        risk_factors.append("Very low credit score (+55)")
    elif credit_score < 670:
        risk_score += 35
        risk_factors.append("Below-average credit score (+35)")
    elif credit_score < 740:
        risk_score += 18
        risk_factors.append("Good credit score (+18)")
    else:
        risk_score += 5
        risk_factors.append("Strong credit score (+5)")

    # Loan-to-income component
    loan_to_income_ratio = loan_amount / annual_income

    if loan_to_income_ratio > 0.80:
        risk_score += 35
        risk_factors.append("Very high loan-to-income ratio (+35)")
    elif loan_to_income_ratio > 0.50:
        risk_score += 22
        risk_factors.append("High loan-to-income ratio (+22)")
    elif loan_to_income_ratio > 0.30:
        risk_score += 12
        risk_factors.append("Moderate loan-to-income ratio (+12)")
    else:
        risk_score += 5
        risk_factors.append("Low loan-to-income ratio (+5)")

    # Employment component
    if employment_years < 1:
        risk_score += 20
        risk_factors.append("Less than one year of employment (+20)")
    elif employment_years < 3:
        risk_score += 10
        risk_factors.append("Limited employment history (+10)")
    else:
        risk_score += 3
        risk_factors.append("Stable employment history (+3)")

    return {
        "risk_score": risk_score,
        "risk_level": (
            "LOW" if risk_score <= 30
            else "MEDIUM" if risk_score <= 45
            else "HIGH"
        ),
        "loan_to_income_ratio": round(loan_to_income_ratio, 3),
        "risk_factors": risk_factors,
    }


def make_loan_decision(
    application: Dict[str, Any],
    risk_score: int,
) -> Dict[str, str]:
    """Apply deterministic approval and rejection rules."""
    age = int(application["age"])

    if age < 18:
        return {
            "decision": "REJECTED",
            "reason": "Applicant must be at least 18 years old.",
        }

    if age < 21:
        return {
            "decision": "REJECTED",
            "reason": "Applicant does not meet the example minimum-age rule of 21.",
        }

    if risk_score <= 45:
        return {
            "decision": "APPROVED",
            "reason": (
                f"Risk score {risk_score} is within the approval "
                "threshold of 45."
            ),
        }

    return {
        "decision": "REJECTED",
        "reason": (
            f"Risk score {risk_score} exceeds the approval "
            "threshold of 45."
        ),
    }


print(get_credit_score("USER1001"))

{'user_id': 'USER1001', 'credit_score': 780, 'source': 'mock_database'}


## 5. Describe the available actions to OpenAI

These function schemas tell the model which actions it may select.

The model can:

1. request missing application fields;
2. request a credit-score lookup;
3. request risk calculation; or
4. request the final loan decision.

The actual functions are executed locally by `act()`.

In [5]:
OPENAI_TOOLS = [
    {
        "type": "function",
        "name": "request_missing_data",
        "description": (
            "Request all required loan-application fields that are currently "
            "missing. Use this whenever missing_fields is not empty."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "fields": {
                    "type": "array",
                    "items": {
                        "type": "string",
                        "enum": [
                            "user_id",
                            "age",
                            "annual_income",
                            "loan_amount",
                            "employment_years",
                        ],
                    },
                    "description": "The application fields that must be supplied.",
                }
            },
            "required": ["fields"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "get_credit_score",
        "description": (
            "Retrieve the applicant's credit score. Use only after all "
            "required application fields are present and no score exists."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "user_id": {
                    "type": "string",
                    "description": "The user ID present in the application.",
                }
            },
            "required": ["user_id"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "calculate_risk",
        "description": (
            "Calculate loan risk. Use only after all required fields and the "
            "credit score are available, and before a risk score exists."
        ),
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "make_loan_decision",
        "description": (
            "Produce the final approval or rejection. Use only after the "
            "risk score has been calculated."
        ),
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False,
        },
        "strict": True,
    },
]

print("Defined actions:", [tool["name"] for tool in OPENAI_TOOLS])

Defined actions: ['request_missing_data', 'get_credit_score', 'calculate_risk', 'make_loan_decision']


## 6. Implement the `Agent` class

The four required methods are:

- `observe()` — inspects current state;
- `reason()` — asks OpenAI to select the next action;
- `act()` — executes the selected local tool;
- `update_state()` — stores the tool result.

A small deterministic validation layer checks that the selected action is valid for the current state. This prevents skipped steps or accidental model hallucinations.

In [6]:
class Agent:
    """A stateful loan-advisor agent coordinated through the OpenAI API."""

    REQUIRED_FIELDS = [
        "user_id",
        "age",
        "annual_income",
        "loan_amount",
        "employment_years",
    ]

    SYSTEM_INSTRUCTIONS = """
You are the reasoning controller for a basic fintech loan-advisor agent.

Choose exactly one next function based only on the supplied state.

Mandatory action policy:
1. If missing_fields is not empty, call request_missing_data with exactly
   those missing fields.
2. Otherwise, if credit_score is null, call get_credit_score.
3. Otherwise, if risk_score is null, call calculate_risk.
4. Otherwise, if decision is null, call make_loan_decision.

Never invent application values.
Never calculate a credit score yourself.
Never calculate risk yourself.
Never approve or reject without first calculating risk.
Do not provide ordinary text; select one function.
"""

    def __init__(
        self,
        client: OpenAI,
        application: Optional[Dict[str, Any]] = None,
        model: str = MODEL,
    ):
        self.client = client
        self.model = model

        self.state: Dict[str, Any] = {
            "application": dict(application or {}),
            "missing_fields": [],
            "credit_score": None,
            "credit_score_source": None,
            "risk_score": None,
            "risk_level": None,
            "loan_to_income_ratio": None,
            "risk_factors": [],
            "decision": None,
            "decision_reason": None,
            "status": "INITIALIZED",
            "iteration": 0,
            "history": [],
        }

    def observe(self) -> Dict[str, Any]:
        """Inspect the application and current internal state."""
        application = self.state["application"]

        missing_fields = [
            field
            for field in self.REQUIRED_FIELDS
            if field not in application or application[field] in (None, "")
        ]

        observation = {
            "application": application.copy(),
            "missing_fields": missing_fields,
            "credit_score": self.state["credit_score"],
            "risk_score": self.state["risk_score"],
            "decision": self.state["decision"],
            "status": self.state["status"],
        }

        self.state["missing_fields"] = missing_fields
        self._record("observe", observation)

        return observation

    def reason(self, observation: Dict[str, Any]) -> Dict[str, Any]:
        """Use the OpenAI API to select exactly one next action."""
        response = self.client.responses.create(
            model=self.model,
            instructions=self.SYSTEM_INSTRUCTIONS,
            input=(
                "Current agent state:\n"
                + json.dumps(observation, indent=2, default=str)
            ),
            tools=OPENAI_TOOLS,
            tool_choice="required",
            parallel_tool_calls=False,
        )

        function_calls = [
            item for item in response.output
            if item.type == "function_call"
        ]

        if len(function_calls) != 1:
            raise RuntimeError(
                "Expected exactly one function call from the model, "
                f"but received {len(function_calls)}."
            )

        function_call = function_calls[0]

        try:
            arguments = json.loads(function_call.arguments)
        except json.JSONDecodeError as exc:
            raise RuntimeError(
                "The model returned invalid JSON function arguments."
            ) from exc

        action = {
            "name": function_call.name,
            "arguments": arguments,
            "call_id": function_call.call_id,
            "response_id": response.id,
        }

        self._validate_selected_action(action, observation)
        self._record(
            "reason",
            {
                "selected_action": action["name"],
                "arguments": action["arguments"],
                "response_id": action["response_id"],
            },
        )

        return action

    def act(
        self,
        action: Dict[str, Any],
        data_provider: Optional[
            Callable[[List[str]], Dict[str, Any]]
        ] = None,
    ) -> Dict[str, Any]:
        """Execute the Python operation selected by the model."""
        action_name = action["name"]
        arguments = action["arguments"]
        application = self.state["application"]

        if action_name == "request_missing_data":
            if data_provider is None:
                raise ValueError(
                    "Missing data was detected, but no data_provider "
                    "was supplied."
                )

            requested_fields = arguments["fields"]
            supplied_data = data_provider(requested_fields)

            if not isinstance(supplied_data, dict):
                raise TypeError(
                    "data_provider must return a dictionary."
                )

            unexpected_fields = (
                set(supplied_data) - set(self.REQUIRED_FIELDS)
            )
            if unexpected_fields:
                raise ValueError(
                    "The provider returned unsupported fields: "
                    f"{sorted(unexpected_fields)}"
                )

            result = {
                "application_update": supplied_data,
                "status": "MISSING_DATA_RECEIVED",
            }

        elif action_name == "get_credit_score":
            # Use the state value rather than trusting a model-supplied ID.
            tool_result = get_credit_score(application["user_id"])
            result = {
                "credit_score": tool_result["credit_score"],
                "credit_score_source": tool_result["source"],
                "status": "CREDIT_SCORE_RETRIEVED",
            }

        elif action_name == "calculate_risk":
            tool_result = calculate_risk(
                application=application,
                credit_score=self.state["credit_score"],
            )
            result = {
                **tool_result,
                "status": "RISK_CALCULATED",
            }

        elif action_name == "make_loan_decision":
            tool_result = make_loan_decision(
                application=application,
                risk_score=self.state["risk_score"],
            )
            result = {
                "decision": tool_result["decision"],
                "decision_reason": tool_result["reason"],
                "status": "COMPLETED",
            }

        else:
            raise ValueError(f"Unsupported action: {action_name}")

        self._record(
            "act",
            {
                "action": action_name,
                "result": result.copy(),
            },
        )

        return result

    def update_state(self, updates: Dict[str, Any]) -> None:
        """Merge an action result into the agent's state."""
        updates = updates.copy()
        application_update = updates.pop("application_update", None)

        if application_update:
            self.state["application"].update(application_update)

        self.state.update(updates)
        self.state["iteration"] += 1

        self._record(
            "update_state",
            {
                "status": self.state["status"],
                "iteration": self.state["iteration"],
                "decision": self.state["decision"],
            },
        )

    def get_summary(self) -> Dict[str, Any]:
        """Return the important final and intermediate values."""
        return {
            "application": self.state["application"],
            "credit_score": self.state["credit_score"],
            "credit_score_source": self.state["credit_score_source"],
            "risk_score": self.state["risk_score"],
            "risk_level": self.state["risk_level"],
            "loan_to_income_ratio": self.state[
                "loan_to_income_ratio"
            ],
            "risk_factors": self.state["risk_factors"],
            "decision": self.state["decision"],
            "decision_reason": self.state["decision_reason"],
            "iterations": self.state["iteration"],
            "status": self.state["status"],
        }

    def _expected_action(
        self,
        observation: Dict[str, Any],
    ) -> str:
        """Return the only valid next action for the current state."""
        if observation["missing_fields"]:
            return "request_missing_data"
        if observation["credit_score"] is None:
            return "get_credit_score"
        if observation["risk_score"] is None:
            return "calculate_risk"
        if observation["decision"] is None:
            return "make_loan_decision"

        raise RuntimeError("The agent already has a final decision.")

    def _validate_selected_action(
        self,
        action: Dict[str, Any],
        observation: Dict[str, Any],
    ) -> None:
        """Guard against a tool call that is inconsistent with state."""
        expected_action = self._expected_action(observation)

        if action["name"] != expected_action:
            raise RuntimeError(
                "Model selected an invalid action. "
                f"Expected {expected_action!r}, "
                f"received {action['name']!r}."
            )

        if action["name"] == "request_missing_data":
            expected_fields = set(observation["missing_fields"])
            selected_fields = set(action["arguments"]["fields"])

            if selected_fields != expected_fields:
                raise RuntimeError(
                    "Model requested the wrong fields. "
                    f"Expected {sorted(expected_fields)}, "
                    f"received {sorted(selected_fields)}."
                )

    def _record(
        self,
        step: str,
        details: Dict[str, Any],
    ) -> None:
        """Append a trace entry for inspection and debugging."""
        self.state["history"].append(
            {
                "step": step,
                "iteration": self.state["iteration"],
                "details": details,
            }
        )


## 7. Implement the agent loop

The loop terminates only when `decision` becomes `APPROVED` or `REJECTED`.

`max_iterations` protects the program from an unintended infinite loop.

In [7]:
def run_agent(
    application: Dict[str, Any],
    data_provider: Optional[
        Callable[[List[str]], Dict[str, Any]]
    ] = None,
    model: str = MODEL,
    max_iterations: int = 10,
    verbose: bool = True,
) -> Agent:
    """Run Observe → Reason → Act → Update until a decision exists."""
    agent = Agent(
        client=client,
        application=application,
        model=model,
    )

    while agent.state["decision"] is None:
        if agent.state["iteration"] >= max_iterations:
            raise RuntimeError(
                f"No decision after {max_iterations} iterations."
            )

        observation = agent.observe()

        if verbose:
            print(
                f"\n{'=' * 18} "
                f"Iteration {agent.state['iteration'] + 1} "
                f"{'=' * 18}"
            )
            print("Observation:")
            pprint(observation)

        action = agent.reason(observation)

        if verbose:
            print("\nOpenAI-selected action:")
            pprint(
                {
                    "name": action["name"],
                    "arguments": action["arguments"],
                }
            )

        result = agent.act(
            action=action,
            data_provider=data_provider,
        )

        if verbose:
            print("\nTool result:")
            pprint(result)

        agent.update_state(result)

    if verbose:
        print("\n" + "=" * 24)
        print("FINAL DECISION")
        print("=" * 24)
        print("Decision:", agent.state["decision"])
        print("Reason:", agent.state["decision_reason"])

    return agent


## 8. Test case A — Complete application expected to be approved

The agent should perform three API-guided iterations:

1. retrieve the credit score;
2. calculate risk;
3. make the decision.

In [8]:
approved_application = {
    "user_id": "USER1001",
    "age": 32,
    "annual_income": 1_200_000,
    "loan_amount": 300_000,
    "employment_years": 6,
}

approved_agent = run_agent(approved_application)

print("\nApplication summary:")
pprint(approved_agent.get_summary())


================== Iteration 1 ==================
Observation:
{'application': {'age': 32,
                 'annual_income': 1200000,
                 'employment_years': 6,
                 'loan_amount': 300000,
                 'user_id': 'USER1001'},
 'credit_score': None,
 'decision': None,
 'missing_fields': [],
 'risk_score': None,
 'status': 'INITIALIZED'}

OpenAI-selected action:
{'arguments': {'user_id': 'USER1001'}, 'name': 'get_credit_score'}

Tool result:
{'credit_score': 780,
 'credit_score_source': 'mock_database',
 'status': 'CREDIT_SCORE_RETRIEVED'}

================== Iteration 2 ==================
Observation:
{'application': {'age': 32,
                 'annual_income': 1200000,
                 'employment_years': 6,
                 'loan_amount': 300000,
                 'user_id': 'USER1001'},
 'credit_score': 780,
 'decision': None,
 'missing_fields': [],
 'risk_score': None,
 'status': 'CREDIT_SCORE_RETRIEVED'}

OpenAI-selected action:
{'arguments': {}, 'name

## 9. Test case B — Application with missing data

This example initially omits `annual_income` and `employment_years`.

The first model-selected action should be `request_missing_data`. The simulated provider returns the missing values, after which the loop continues automatically.

In [9]:
incomplete_application = {
    "user_id": "USER1004",
    "age": 27,
    "loan_amount": 250_000,
}

simulated_user_responses = {
    "annual_income": 900_000,
    "employment_years": 4,
}


def simulated_data_provider(
    requested_fields: List[str],
) -> Dict[str, Any]:
    """Simulate an applicant supplying requested information."""
    print("\nApplicant was asked for:", requested_fields)

    return {
        field: simulated_user_responses[field]
        for field in requested_fields
        if field in simulated_user_responses
    }


missing_data_agent = run_agent(
    application=incomplete_application,
    data_provider=simulated_data_provider,
)

print("\nApplication summary:")
pprint(missing_data_agent.get_summary())


================== Iteration 1 ==================
Observation:
{'application': {'age': 27, 'loan_amount': 250000, 'user_id': 'USER1004'},
 'credit_score': None,
 'decision': None,
 'missing_fields': ['annual_income', 'employment_years'],
 'risk_score': None,
 'status': 'INITIALIZED'}

OpenAI-selected action:
{'arguments': {'fields': ['annual_income', 'employment_years']},
 'name': 'request_missing_data'}

Applicant was asked for: ['annual_income', 'employment_years']

Tool result:
{'application_update': {'annual_income': 900000, 'employment_years': 4},
 'status': 'MISSING_DATA_RECEIVED'}

================== Iteration 2 ==================
Observation:
{'application': {'age': 27,
                 'annual_income': 900000,
                 'employment_years': 4,
                 'loan_amount': 250000,
                 'user_id': 'USER1004'},
 'credit_score': None,
 'decision': None,
 'missing_fields': [],
 'risk_score': None,
 'status': 'MISSING_DATA_RECEIVED'}

OpenAI-selected action:
{'

## 10. Test case C — Complete application expected to be rejected

The mock credit score is low, the requested loan is large relative to income, and employment history is short.

In [10]:
rejected_application = {
    "user_id": "USER1003",
    "age": 29,
    "annual_income": 500_000,
    "loan_amount": 450_000,
    "employment_years": 0.5,
}

rejected_agent = run_agent(rejected_application)

print("\nApplication summary:")
pprint(rejected_agent.get_summary())


================== Iteration 1 ==================
Observation:
{'application': {'age': 29,
                 'annual_income': 500000,
                 'employment_years': 0.5,
                 'loan_amount': 450000,
                 'user_id': 'USER1003'},
 'credit_score': None,
 'decision': None,
 'missing_fields': [],
 'risk_score': None,
 'status': 'INITIALIZED'}

OpenAI-selected action:
{'arguments': {'user_id': 'USER1003'}, 'name': 'get_credit_score'}

Tool result:
{'credit_score': 540,
 'credit_score_source': 'mock_database',
 'status': 'CREDIT_SCORE_RETRIEVED'}

================== Iteration 2 ==================
Observation:
{'application': {'age': 29,
                 'annual_income': 500000,
                 'employment_years': 0.5,
                 'loan_amount': 450000,
                 'user_id': 'USER1003'},
 'credit_score': 540,
 'decision': None,
 'missing_fields': [],
 'risk_score': None,
 'status': 'CREDIT_SCORE_RETRIEVED'}

OpenAI-selected action:
{'arguments': {}, 'na

## 11. Optional interactive missing-data provider

Use this provider when you want the notebook user to type missing values directly in Colab.

In [11]:
def interactive_data_provider(
    requested_fields: List[str],
) -> Dict[str, Any]:
    """Collect missing application values from notebook input."""
    converters = {
        "user_id": str,
        "age": int,
        "annual_income": float,
        "loan_amount": float,
        "employment_years": float,
    }

    collected_data: Dict[str, Any] = {}

    for field in requested_fields:
        raw_value = input(f"Enter {field}: ").strip()

        try:
            collected_data[field] = converters[field](raw_value)
        except ValueError as exc:
            raise ValueError(
                f"Invalid value for {field}: {raw_value!r}"
            ) from exc

    return collected_data


# Example: uncomment and run.
#
# interactive_application = {
#     "user_id": "USER1002",
#     "loan_amount": 350_000,
# }
#
# interactive_agent = run_agent(
#     application=interactive_application,
#     data_provider=interactive_data_provider,
# )
#
# pprint(interactive_agent.get_summary())

## 12. Inspect the agent trace

Every Observe–Reason–Act–Update step is retained in `state["history"]`.

Run the cell after one of the test cases.

In [12]:
for event in missing_data_agent.state["history"]:
    print(
        f"Iteration {event['iteration']} | "
        f"{event['step'].upper()}"
    )
    pprint(event["details"])
    print("-" * 70)

Iteration 0 | OBSERVE
{'application': {'age': 27, 'loan_amount': 250000, 'user_id': 'USER1004'},
 'credit_score': None,
 'decision': None,
 'missing_fields': ['annual_income', 'employment_years'],
 'risk_score': None,
 'status': 'INITIALIZED'}
----------------------------------------------------------------------
Iteration 0 | REASON
{'arguments': {'fields': ['annual_income', 'employment_years']},
 'response_id': 'resp_061768856918e2b0006a532d612eb0819dab210b46648d1757',
 'selected_action': 'request_missing_data'}
----------------------------------------------------------------------
Iteration 0 | ACT
{'action': 'request_missing_data',
 'result': {'application_update': {'annual_income': 900000,
                                   'employment_years': 4},
            'status': 'MISSING_DATA_RECEIVED'}}
----------------------------------------------------------------------
Iteration 1 | UPDATE_STATE
{'decision': None, 'iteration': 1, 'status': 'MISSING_DATA_RECEIVED'}
---------------------

## How the requirements are satisfied

| Requirement | Implementation |
|---|---|
| Takes a loan application | Application dictionary passed to `Agent` |
| Calls a credit-score tool | `get_credit_score()` mock Python function |
| Calculates risk | `calculate_risk()` deterministic Python function |
| Approves or rejects | `make_loan_decision()` |
| Uses OpenAI API | `reason()` calls `client.responses.create()` |
| `observe()` | Finds missing fields and reads state |
| `reason()` | Model selects the next function |
| `act()` | Routes the function call to Python |
| `update_state()` | Merges tool results into state |
| No framework | Uses only plain Python and OpenAI SDK |
| Handles missing data | `request_missing_data` plus provider callback |
| Iterates to completion | `run_agent()` loop |
| Traceability | Every step is stored in `history` |

## Key design decision

OpenAI performs **orchestration**, not the numerical credit decision.

This makes the workflow agentic while keeping:

- application values grounded in user input;
- credit scores grounded in the mock tool;
- risk calculations reproducible;
- approval rules deterministic; and
- all intermediate actions auditable.

## Possible extensions

- Add debt, repayment history, loan term, and collateral fields.
- Validate field ranges before credit lookup.
- Replace the mock score with a sandbox credit-bureau API.
- Add a separate explanation-generation step after the deterministic decision.
- Persist state in a database.
- Add retry handling for API timeouts and rate limits.
- Add human review for borderline applications.
- Add unit tests for every tool and state transition.

## References

- OpenAI function-calling guide: https://developers.openai.com/api/docs/guides/function-calling
- OpenAI Responses API reference: https://platform.openai.com/docs/api-reference/responses